In [1]:
!pip install -q kagglehub

import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [2]:
path = kagglehub.dataset_download("jessicali9530/kuc-hackathon-winter-2018")
print("Downloaded to:", path)

import os
tsv_candidates = [os.path.join(root, f) for root, _, files in os.walk(path) for f in files if f.endswith(".tsv")]
print("TSV files found:", tsv_candidates)

100%|██████████| 40.7M/40.7M [00:00<00:00, 154MB/s]

Extracting files...


Downloaded to: /root/.cache/kagglehub/datasets/jessicali9530/kuc-hackathon-winter-2018/versions/2
TSV files found: []


In [3]:
import os

print("path variable is:", path)
print("Does path exist?", os.path.exists(path))
print()
print("ALL files found under path (any extension):")
for root, dirs, files in os.walk(path):
    for f in files:
        full = os.path.join(root, f)
        size_mb = os.path.getsize(full) / (1024 * 1024)
        print(f"  {full}  ({size_mb:.1f} MB)")

path variable is: /root/.cache/kagglehub/datasets/jessicali9530/kuc-hackathon-winter-2018/versions/2
Does path exist? True

ALL files found under path (any extension):
  /root/.cache/kagglehub/datasets/jessicali9530/kuc-hackathon-winter-2018/versions/2/drugsComTest_raw.csv  (26.4 MB)
  /root/.cache/kagglehub/datasets/jessicali9530/kuc-hackathon-winter-2018/versions/2/drugsComTrain_raw.csv  (79.1 MB)


In [4]:
csv_candidates = [os.path.join(root, f) for root, _, files in os.walk(path) for f in files if f.endswith(".csv")]
print("CSV files found:", csv_candidates)

train_path = next(p for p in csv_candidates if "Train" in p)
test_path = next(p for p in csv_candidates if "Test" in p)

with open(train_path) as f:
    first_line = f.readline()
print("Raw first line:")
print(repr(first_line))

CSV files found: ['/root/.cache/kagglehub/datasets/jessicali9530/kuc-hackathon-winter-2018/versions/2/drugsComTest_raw.csv', '/root/.cache/kagglehub/datasets/jessicali9530/kuc-hackathon-winter-2018/versions/2/drugsComTrain_raw.csv']
Raw first line:
'uniqueID,drugName,condition,review,rating,date,usefulCount\n'


In [5]:
import pandas as pd

def load_with_detected_delimiter(path):
    with open(path) as f:
        first_line = f.readline()
    delimiter = "\t" if first_line.count("\t") > first_line.count(",") else ","
    df = pd.read_csv(path, sep=delimiter)
    return df, delimiter

train_df, delim = load_with_detected_delimiter(train_path)
print(f"Detected delimiter: {repr(delim)}")
print("Train columns:", train_df.columns.tolist())
print("Train shape:", train_df.shape)

# Sanity check: did we actually get the real columns, or one giant garbage column?
expected_cols = {"drugName", "condition", "rating", "date", "usefulCount"}
if not expected_cols.issubset(set(train_df.columns)):
    raise ValueError(
        f"Delimiter detection likely failed — got columns {train_df.columns.tolist()}, "
        f"expected something containing {expected_cols}. Check the raw first line above."
    )

test_df, _ = load_with_detected_delimiter(test_path)

df = pd.concat([train_df, test_df], ignore_index=True)
print()
print("Combined shape:", df.shape)
print("Null conditions:", df["condition"].isna().sum())

df = df.dropna(subset=["condition", "drugName"]).reset_index(drop=True)
print("After dropping nulls:", df.shape)

Detected delimiter: ','
Train columns: ['uniqueID', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount']
Train shape: (161297, 7)

Combined shape: (215063, 7)
Null conditions: 1194
After dropping nulls: (213869, 7)


In [6]:
df["condition_len"] = df["condition"].str.len()

print("Condition length distribution:")
print(df["condition_len"].describe())
print()
print("Longest condition strings (inspect these for garbled/junk data):")
print(df.drop_duplicates(subset="condition").nlargest(10, "condition_len")[["condition", "condition_len"]].to_string())

Condition length distribution:
count    213869.000000
mean         14.000201
std           7.024055
min           2.000000
25%          10.000000
50%          13.000000
75%          18.000000
max          67.000000
Name: condition_len, dtype: float64

Longest condition strings (inspect these for garbled/junk data):
                                                                  condition  condition_len
9627    Prosthetic Heart Valves, Mechanical Valves - Thrombosis Prophylaxis             67
9807        Prosthetic Heart Valves, Tissue Valves - Thrombosis Prophylaxis             63
23898       Deep Vein Thrombosis Prophylaxis after Knee Replacement Surgery             63
21787        Deep Vein Thrombosis Prophylaxis after Hip Replacement Surgery             62
2942             Vitamin/Mineral Supplementation during Pregnancy/Lactation             58
40180             Chronic Inflammatory Demyelinating Polyradiculoneuropathy             57
83225               Hyperlipoproteinemia Type 

In [13]:
MIN_REVIEWS = 5

before = len(per_drug)
per_drug = per_drug[per_drug["review_count"] >= MIN_REVIEWS].reset_index(drop=True)
print(f"Filtered out {before - len(per_drug)} (condition, drug) pairs with fewer than {MIN_REVIEWS} reviews")
print(f"Remaining: {len(per_drug)} pairs across {per_drug['condition'].nunique()} conditions")

Filtered out 5895 (condition, drug) pairs with fewer than 5 reviews
Remaining: 3551 pairs across 454 conditions


In [18]:
import re

junk_pattern = re.compile(r"[<>]")

junk_keys = [k for k in rankings if junk_pattern.search(k)]
print(f"Removing {len(junk_keys)} junk condition(s) from rankings")

for k in junk_keys:
    del rankings[k]

print(f"Clean rankings: {len(rankings)} real conditions remain")

Removing 5 junk condition(s) from rankings
Clean rankings: 449 real conditions remain


In [17]:
junk_mask = df["condition"].str.contains("<|>", regex=True, na=False)
print(f"Rows with HTML-tag-like conditions: {junk_mask.sum()}")
print()
print("Unique junk condition values found:")
print(df.loc[junk_mask, "condition"].value_counts())

Rows with HTML-tag-like conditions: 1171

Unique junk condition values found:
condition
0</span> users found this comment helpful.      128
1</span> users found this comment helpful.      114
2</span> users found this comment helpful.      105
3</span> users found this comment helpful.      101
4</span> users found this comment helpful.       87
                                               ... 
70</span> users found this comment helpful.       1
100</span> users found this comment helpful.      1
135</span> users found this comment helpful.      1
38</span> users found this comment helpful.       1
105</span> users found this comment helpful.      1
Name: count, Length: 80, dtype: int64


In [21]:
TOP_K_PER_CONDITION = 15

rankings = {}
for condition, group in per_drug.groupby("condition"):
    top = group.sort_values("score", ascending=False).head(TOP_K_PER_CONDITION)
    rankings[condition] = [
        {"drug": row["drugName"], "score": float(row["score"]), "review_count": int(row["review_count"])}
        for _, row in top.iterrows()
    ]

print(f"{len(rankings)} conditions ready for export")
# Spot check one
example_condition = list(rankings.keys())[0]
print(f"\nExample — {example_condition}:")
for entry in rankings[example_condition][:5]:
    print(" ", entry)

454 conditions ready for export

Example — 0</span> users found this comment helpful.:
  {'drug': 'Depo-Provera', 'score': 7.14, 'review_count': 8}
  {'drug': 'Implanon', 'score': 6.73, 'review_count': 7}
  {'drug': 'Microgestin Fe 1 / 20', 'score': 5.8, 'review_count': 5}
  {'drug': 'Loestrin 24 Fe', 'score': 5.78, 'review_count': 24}
  {'drug': 'Mirena', 'score': 5.57, 'review_count': 7}


In [22]:
import json, os

os.makedirs("data_artifacts", exist_ok=True)
with open("data_artifacts/drug_rankings.json", "w") as f:
    json.dump(rankings, f, indent=2)

print("Exported drug_rankings.json")
!ls -la data_artifacts

Exported drug_rankings.json
total 240
drwxr-xr-x 2 root root   4096 Sep 26 17:33 .
drwxr-xr-x 1 root root   4096 Sep 26 17:33 ..
-rw-r--r-- 1 root root    438 Sep 26 17:33 contraindication_rules.json
-rw-r--r-- 1 root root 230239 Sep 26 17:44 drug_rankings.json


In [23]:
contraindication_rules = {
    "allergy_rules": {
        # Well-known cross-reactivity classes — illustrative only, NOT exhaustive.
        "penicillin": ["amoxicillin", "ampicillin", "penicillin v potassium"],
        "sulfa": ["sulfamethoxazole", "sulfasalazine"],
        "nsaid": ["ibuprofen", "naproxen", "aspirin"],
    },
    "interaction_rules": {
        # Well-known, commonly-cited interaction pairs — illustrative only.
        "warfarin": ["ibuprofen", "aspirin", "naproxen"],
        "maoi": ["sertraline", "fluoxetine"],
    },
}

with open("data_artifacts/contraindication_rules.json", "w") as f:
    json.dump(contraindication_rules, f, indent=2)

print("Exported contraindication_rules.json (illustrative only — see service README)")

Exported contraindication_rules.json (illustrative only — see service README)


In [24]:
import shutil
shutil.make_archive("drug_recommendation_artifacts", "zip", "data_artifacts")

from google.colab import files
files.download("drug_recommendation_artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>